# Overview: Trích xuất lược đồ cho câu hỏi

Các vấn đề cần quan tâm trong một câu hỏi của user, có thể bao gồm:
- Các keywords quan trọng -> dùng cho mục đích retrieval
- Các câu hỏi của người dùng -> nhằm mục đích bot trả lời chuẩn chỉnh hơn, trọng tâm hơn. Trong trường hợp câu hỏi đầu vào không xác định -> giúp chatbot biết rằng nên hỏi lại người dùng hoặc phản hồi tự nhiên hơn (tránh không hỏi cũng ép trả lời về lịch sử)
- Lược đồ với ví dụ bằng trình giữ chỗ

# Define LLM

In [47]:
import os
import getpass
from langchain_cohere import ChatCohere
from langchain.chat_models import init_chat_model

if not os.environ.get("COHERE_API_KEY"):
    os.environ["COHERE_API_KEY"] = getpass.getpass("Enter Cohere API key:")

llm = init_chat_model("command-a-03-2025", model_provider="cohere")

# Kế hoạch xây dựng lược đồ

## Trích xuất thông tin cá nhân người dùng

Mục đích: Dùng để cá nhân hóa chatbot cho người dùng. Các thông tin có thể trích xuất bao gồm:
- Tên người dùng: "Tôi tên là...", "Gọi tôi là..."
- Tuổi: "Tôi học lớp 8", "Tôi 24 tuổi"
- Ngôn ngữ: Ngôn ngữ dùng để đặt câu hỏi
- Vị trí (quê quán): "Tôi sống ở Huế", "Người miền Bắc"
- Trình độ học vấn: "Em không rành mấy về sử", "Tôi thích lịch sử thế kỷ 20"
- Mức độ kiến thức lịch sử: "Em không rành mấy về sử", "Tôi thích lịch sử thế kỷ 20"
- Sở thích: "Em mê sử chiến tranh", "Tôi thích vua Quang Trung"
- Phong cách trả lời mong muốn: "Nói hài hước tí nha", "Em muốn giải thích kỹ càng"
- Cảm xúc: Cảm xúc hiện tại có thể suy luận được

=> Điều chỉnh giọng văn, độ sâu câu trả lời và ví dụ phù hợp

## Trích xuất thông tin liên quan đến mối quan tâm

Mục đích: Dùng để theo dõi mạch hội thoại và gợi ý tiếp theo đúng hướng. Các thông trích có thể trích xuất báo gồm:
- Chủ đề đang hỏi: "Trận Điện Biên Phủ", "Chiến tranh thế giới"
- Giai đoạn lịch sử: "thời Lê sơ", "thế kỷ 19"
- Nhân vật quan tâm: "Hồ Chí Minh", "Nguyễn Huệ"
- Câu hỏi cụ thể chưa được trả lời hết: "Hồi nãy bạn nói tới triều Nguyễn..."
- Cảm xúc người dùng: "Em thấy buồn cười", "Nghe hơi ghê", "Sao ác vậy ta?"

=> Những thông tin này giúp chatbot nối tiếp mạch hội thoại đúng phản ứng tự nhiên và lưu ngữ cảnh phù hợp

## Trích xuất thông tin lịch sử cuộc trò chuyện

Mục đích: Dùng để tái hiện trí nhớ tạm thời hoặc lâu dài. Các thông tin có thể trích xuất bao gồm:
- Chủ đề từng nói: Lưu vào session context (hoặc vector DB nếu dài)
- Quan điểm người dùng: Ví dụ: “Em nghĩ thực dân Pháp ác lắm” → gợi mở góc nhìn phản biện
- Hỏi lại: “Bạn có thể nói rõ hơn trận đó hồi nãy?”

## Workflow tổng quan

[1]: User input

[2]: Information extraction

[3]: User memory manager: Update profile

[4]: Conversation memory

[5]: Query analyzer: Phân tích cần dùng RAG hay không. Nếu cần, thực hiện retrieval các thông tin đã trích xuất được

[6]: Response generator: Kết hợp với các thông tin trên để sinh câu trả lời

=> Kết quả: Câu trả lời có tính cá nhân hóa


# Import

In [37]:
import os
import time
import warnings
warnings.filterwarnings("ignore")

import uuid
import getpass
import enum
from enum import Enum
from typing import Dict
from typing import List
from typing import Type
from typing import TypedDict
from typing import Optional

from pydantic import BaseModel
from pydantic import Field

from langchain_cohere import ChatCohere
from langchain.chat_models import init_chat_model

from langchain_core.messages import AIMessage
from langchain_core.messages import HumanMessage
from langchain_core.messages import BaseMessage
from langchain_core.messages import SystemMessage
from langchain_core.messages import ToolMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import MessagesPlaceholder


# Trích xuất thông tin cá nhân người dùng

## Định nghĩa lược đồ

In [49]:
class Person(BaseModel):
    """Trích xuất các thông tin cá nhân người dùng từ câu hỏi được người dùng nhập vào!"""
    # Default hoặc dùng ... để đánh dấu là bắt buộc
    # name: Optional[str] = Field(default=None, description="")
    name: Optional[str] = Field(..., description="Tên của người đặt câu hỏi.")
    age_group: Optional[str] = Field(
        ...,
        description="Nhóm tuổi của người dùng nếu được cung cấp hoặc có thể suy ra được.",
        enum=["child", "teen", "adult", "senior"]
    )
    language: Optional[str] = Field(
        ...,
        description="Ngôn ngữ hiện tại mà người dùng đang đặt câu hỏi.",
        enum=["vietnamese", "english", "french", "chinese"]
    )
    # region: Optional[str] = Field(
    #     ...,
    #     # default=None,
    #     description="Vùng miền hoặc quốc gia người dùng đến từ nếu được nhắc tới (ví dụ: miền Trung, Hà Nội, Nhật Bản)."
    # ),
    # level: Optional[Level] = Field(
    #     ...,
    #     # default=None,
    #     description="Trình độ kiến thức lịch sử của người dùng (beginner, intermediate, advanced).",
    # ),
    # interests: Optional[List[str]] = Field(
    #     ...,
    #     # default=None,
    #     description="Danh sách các chủ đề lịch sử mà người dùng bày tỏ sự quan tâm. Loại trừ các chủ đề được nhắc chung chung hoặc vô tình.",
    # )
    # tone_preference: Optional[str] = Field(
    #     ...,
    #     # default=None,
    #     description="Phong cách phản hồi người dùng ưa thích, ví dụ: hài hước, nghiêm túc, thân mật, học thuật.",
    # )
    # current_emotion: Optional[str] = Field(
    #     ...,
    #     # default=None,
    #     description="Cảm xúc hiện tại nếu có thể suy luận được, ví dụ: tò mò, buồn, vui, thất vọng.",
    # )

## Định nghĩa một model từ đó có thể trích xuất đa thực thể

In [50]:
class People(BaseModel):
    """Extract data about people."""
    people: List[Person]

## Định nghĩa các ví dụ tham chiếu

In [51]:
class Example(TypedDict):
    """A representation of an example consisting of text input and expected tool calls.

    For extraction, the tool calls are represented as instances of pydantic model.
    """

    input: str  # This is the example text
    tool_calls: List[BaseModel]  # Instances of pydantic model that should be extracted


def tool_example_to_messages(example: Example) -> List[BaseMessage]:
    """Convert an example into a list of messages that can be fed into an LLM.

    This code is an adapter that converts our example to a list of messages
    that can be fed into a chat model.

    The list of messages per example corresponds to:

    1) HumanMessage: contains the content from which content should be extracted.
    2) AIMessage: contains the extracted information from the model
    3) ToolMessage: contains confirmation to the model that the model requested a tool correctly.

    The ToolMessage is required because some of the chat models are hyper-optimized for agents
    rather than for an extraction use case.
    """
    messages: List[BaseMessage] = [HumanMessage(content=example["input"])]
    tool_calls = []
    for tool_call in example["tool_calls"]:
        tool_calls.append(
            {
                "id": str(uuid.uuid4()),
                "args": tool_call.model_dump(),
                # The name of the function right now corresponds
                # to the name of the pydantic model
                # This is implicit in the API right now,
                # and will be improved over time.
                "name": tool_call.__class__.__name__,
            },
        )
    messages.append(AIMessage(content="", tool_calls=tool_calls))
    tool_outputs = example.get("tool_outputs") or [
        "You have correctly called this tool."
    ] * len(tool_calls)
    for output, tool_call in zip(tool_outputs, tool_calls):
        messages.append(ToolMessage(content=output, tool_call_id=tool_call["id"]))
    return messages

## Định nghĩa các ví dụ hướng dẫn

In [52]:
# Định nghĩa các ví dụ hướng dẫn và chuyển chúng thành định dạng mesages
examples = [
    (
        "Hãy kể về lịch sử Đinh Bộ Lĩnh dẹp loạn 12 sứ quân",
        People(people=[]),
    ),
    (
        "Mình là Nam, mình rất thích tìm hiểu về lịch sử Việt Nam.",
        People(people=[Person(name="Nam", age_group=None, language="vietnamese")])
    )
]

messages = []

for text, tool_call in examples:
    messages.extend(tool_example_to_messages({"input": text, "tool_calls": [tool_call]}))

## Xây dựng prompt

### Prompt cơ bản

In [23]:
# system_prompt = """
#     Bạn là một chuyên gia trong việc trích xuất các thông tin.
#     Hãy chỉ trích xuất thông tin liên quan từ văn bản.
#     Nếu bạn không biết thông tin/giá trị được yêu cầu hãy trả về giá trị null cho những thông tin/giá trị không được cung cấp.
# """

# messages = [
#     ("system", system_prompt),
#     ("human", "{text}")
# ]

# prompt = ChatPromptTemplate.from_messages(messages)

### Prompt có ví dụ bằng trình giữ chỗ

Việc xây dựng một prompt hỗ trợ trích xuất có một trình giữ chỗ nhằm cung cấp ví dụ để cải thiện các thông tin trích xuất.

In [53]:
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are an expert extraction algorithm. "
            "Only extract relevant information from the text. "
            "If you do not know the value of an attribute asked "
            "to extract, return null for the attribute's value.",
        ),
        MessagesPlaceholder("examples"),
        ("human", "{text}"),
    ]
)

## Chain runnable trích xuất

In [54]:
runnable = prompt | llm.with_structured_output(
    schema=People,
    method="function_calling",
)

## Trích xuất lược đồ mà không cung cấp ví dụ tham chiếu

In [55]:
text = "Lịch sử Việt Nam thật là hào hùng, đúng không nhỉ. Mình là Nam, mình rất thích học lịch sử"
output = runnable.invoke({"text": text, "examples": messages})

In [56]:
print(output)

None
